# ETL data validation

This notebook audits the automated Adzuna snapshots and validates the tables prepared for Power BI. The main fact-table grain is one job observed on one snapshot date.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

REPOSITORY_URL = 'https://github.com/italofvaz/european-job-market-analysis.git'

if Path('/content').exists():
    PROJECT_ROOT = Path('/content/european-job-market-analysis')
    if PROJECT_ROOT.exists():
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
else:
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / 'data').exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
from src.transform_powerbi import build_powerbi_tables, load_sources, run_etl, validate_tables

print('Project root:', PROJECT_ROOT)

## 1. Load the source layers

In [ ]:
sources = load_sources(PROJECT_ROOT)

source_inventory = pd.DataFrame([
    {
        'dataset': name,
        'rows': len(frame),
        'columns': len(frame.columns),
    }
    for name, frame in sources.items()
]).sort_values('dataset')

display(source_inventory)

## 2. Validate table grain and keys

`jobs_master` must contain one row per job. Daily snapshots must contain one row per job and snapshot date.

In [ ]:
daily = sources['daily']
master = sources['master']
matches = sources['matches']

key_checks = pd.DataFrame([
    {
        'check': 'Duplicate job_key in jobs_master',
        'issues': master.duplicated('job_key').sum(),
    },
    {
        'check': 'Duplicate job and date in daily snapshots',
        'issues': daily.duplicated(['job_key', 'snapshot_date']).sum(),
    },
    {
        'check': 'Duplicate search-match grain',
        'issues': matches.duplicated(
            ['job_key', 'snapshot_date', 'role_family', 'search_term']
        ).sum(),
    },
])
key_checks['status'] = key_checks['issues'].eq(0).map({True: 'PASS', False: 'FAIL'})
display(key_checks)

## 3. Review snapshot coverage

In [ ]:
snapshot_coverage = (
    daily.groupby(['snapshot_date', 'country'])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)
display(snapshot_coverage)

## 4. Review missing values

Missing location coordinates, salary, contract information, or company names remain visible. The ETL does not invent replacements for unavailable source data.

In [ ]:
important_columns = [
    'job_title', 'company', 'city', 'region', 'latitude', 'longitude',
    'published_at', 'salary_min', 'salary_max', 'contract_type',
    'contract_time', 'job_description', 'job_url',
]

missing_summary = pd.DataFrame({
    'column': important_columns,
    'missing_rows': [master[column].isna().sum() for column in important_columns],
    'missing_percentage': [master[column].isna().mean() * 100 for column in important_columns],
}).sort_values('missing_percentage', ascending=False)

missing_summary['missing_percentage'] = missing_summary['missing_percentage'].round(1)
display(missing_summary)

## 5. Build and validate the Power BI tables

Salary values below 10,000 or above 300,000 are preserved but excluded from annual comparisons. These thresholds are analytical quality rules, not corrections to the source values.

In [ ]:
tables = build_powerbi_tables(sources)
etl_checks = validate_tables(tables)
display(etl_checks)

salary_quality = (
    tables['fact_job_snapshots']
    .groupby(['country_key', 'salary_quality_status'])
    .size()
    .reset_index(name='job_snapshots')
    .sort_values(['country_key', 'job_snapshots'], ascending=[True, False])
)
display(salary_quality)

## 6. Review extracted skills

Skill mentions come from the available job title and description snippet. Frequencies can therefore be understated.

In [ ]:
skill_summary = (
    tables['bridge_job_skills']
    .merge(tables['dim_skill'], on='skill_key', how='left', validate='many_to_one')
    .groupby(['skill_category', 'skill_name'])['job_key']
    .nunique()
    .reset_index(name='jobs_mentioning_skill')
    .sort_values('jobs_mentioning_skill', ascending=False)
)
display(skill_summary)

## 7. Save the processed layer

This command writes the validated CSV tables to `data/processed`. The GitHub Actions workflow runs the same command after each daily collection.

In [ ]:
processed_tables = run_etl(PROJECT_ROOT)

processed_inventory = pd.DataFrame([
    {'table': name, 'rows': len(frame), 'columns': len(frame.columns)}
    for name, frame in processed_tables.items()
]).sort_values('table')
display(processed_inventory)